In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
BATCH_SIZE = 128
EPOCHS = 30 # Increased epochs for better convergence with augmentation
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Found {torch.cuda.device_count()} GPUs!")

In [ ]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(), # Convert numpy array to PIL Image
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(), # Convert back to tensor and normalize to [0, 1]
])

# --- 3. Modified Custom Dataset ---
class DigitDataset(Dataset):
    def __init__(self, data, labels=None, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Reshape to 28x28, keep as uint8 for PIL transformations
        image = self.data[idx].reshape(28, 28).astype(np.uint8)

        if self.transform:
            # Apply augmentations (which includes ToTensor and normalization)
            image = self.transform(image)
        else:
            # For validation/test set: just normalize and convert to tensor
            image = torch.tensor(image / 255.0, dtype=torch.float32).unsqueeze(0)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return image, label
        else:
            return image

In [ ]:
class DigitRecognizerCNN_V2(nn.Module):
    def __init__(self):
        super(DigitRecognizerCNN_V2, self).__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2), nn.Dropout(0.25)
        )
        self.fc_stack = nn.Sequential(
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_stack(x)
        x = x.view(x.size(0), -1)
        x = self.fc_stack(x)
        return x

In [ ]:
from sklearn.model_selection import train_test_split
if __name__ == "__main__":
    train_df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
    test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

    X_full = train_df.drop("label", axis=1).values
    y_full = train_df["label"].values
    X_test = test_df.values

    # Split data *before* creating Datasets
    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=VALIDATION_SPLIT, random_state=42
    )

    # Create datasets: apply transforms ONLY to the training set
    train_dataset = DigitDataset(X_train, y_train, transform=train_transforms)
    val_dataset = DigitDataset(X_val, y_val, transform=None) # No augmentation for validation

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Instantiate the new, deeper model
    model = DigitRecognizerCNN_V2()

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for Data Parallelism.")
        model = nn.DataParallel(model)
    model.to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # --- NEW: Initialize the Learning Rate Scheduler ---
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=2, verbose=True)

    print("\nStarting training with enhanced model and augmentations...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        accuracy = 100 * correct / total
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Accuracy: {accuracy:.2f}%")

        # --- NEW: Step the scheduler based on validation loss ---
        scheduler.step(avg_val_loss)

    print("Finished training.\n")
    print("Generating submission file...")
    
    test_dataset = DigitDataset(X_test, transform=None)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    predictions = []
    model.eval()
    with torch.no_grad():
        for images in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            predictions.extend(predicted.cpu().numpy())

    submission_df = pd.DataFrame({"ImageId": range(1, len(predictions) + 1), "Label": predictions})
    submission_df.to_csv("submission.csv", index=False)
    print("submission.csv created successfully!")
